# Credit Risk Early Warning System
## Notebook 01 — Exploratory Data Analysis

**Objective:**  
Understand which borrower characteristics drive loan default — and build the business case for every modeling decision that follows.

**Story this notebook tells:**  
A meaningful fraction of loans in this portfolio end in default. We investigate *who* defaults, *why* they default, and *what signals* a credit officer could have used to catch them earlier.

**Reader takeaway:**  
By the end of this notebook, the features selected for modeling should feel inevitable — not arbitrary.

---

| | |
|---|---|
| **Dataset** | LendingClub Loan Data (2007–2020) |
| **Target** | `loan_status` → Charged Off = 1, Fully Paid = 0 |
| **Exclusions** | Current, Late, In Grace Period (outcome unknown) |

---
## Section 0 — Setup and Data Loading

In [3]:
import sys
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Add src/ to path so we can import our modules directly
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

from preprocessing import prepare_data
from features import create_all_features

# ── Visual style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

# Consistent color palette throughout this notebook
COLOR_DEFAULT    = '#c0392b'   # muted red  → defaulted loans
COLOR_NONDEFAULT = '#2980b9'   # steel blue → fully paid loans
COLOR_NEUTRAL    = '#7f8c8d'   # grey       → neutral / combined views
PALETTE_RISK     = 'Reds'      # sequential → risk gradient charts

print('Setup complete.')

Setup complete.


### Load Raw Data

The loader handles both `.csv` and `.csv.gz` files and does not assume a fixed filename — it picks up whatever LendingClub file is present in `data/` or `datasets/`.

In [4]:
def load_lending_club_data(search_dirs=None):
    """
    Locate and load the LendingClub CSV (plain or gzip-compressed).
    Searches the provided directories in order and returns the first match.
    """
    if search_dirs is None:
        base = os.path.join(os.getcwd(), '..')
        search_dirs = [
            os.path.join(base, 'datasets'),
            os.path.join(base, 'data'),
            base,
        ]

    for directory in search_dirs:
        for pattern in ['*.csv', '*.csv.gz']:
            matches = glob.glob(os.path.join(directory, pattern))
            if matches:
                path = matches[0]
                print(f'Loading: {path}')
                df = pd.read_csv(path, low_memory=False)
                print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
                return df

    raise FileNotFoundError(
        'No CSV or CSV.GZ file found. Place the LendingClub dataset in datasets/ or data/.'
    )


raw_df = load_lending_club_data()

FileNotFoundError: No CSV or CSV.GZ file found. Place the LendingClub dataset in datasets/ or data/.

### Preprocess and Engineer Features

In [ ]:
output_dir = os.path.join(os.getcwd(), '..', 'outputs')

# Run full preprocessing pipeline — cleans types, removes leakage,
# creates target variable, saves data quality report
clean_df = prepare_data(raw_df.copy(), output_dir=output_dir)

# Engineer banking-domain features on top of clean data
df = create_all_features(clean_df.copy(), output_dir=output_dir)

print(f'\nFinal analytical dataset: {df.shape[0]:,} loans × {df.shape[1]} features')

---
## Section 1 — Data Quality Audit

Before any analysis, we confirm the data is usable. A model built on dirty data produces unreliable risk scores — so data quality is not a formality here, it is a prerequisite.

In [ ]:
# Basic shape and composition
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\nData types:')
print(df.dtypes.value_counts())

### 1.1 Missing Values

In [ ]:
missing = (
    df.isnull().mean()
    .mul(100)
    .round(2)
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_pct'})
    .query('missing_pct > 0')
    .sort_values('missing_pct', ascending=False)
)

print(f'Columns with missing values: {len(missing)}')
print(missing.to_string(index=False))

In [ ]:
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(9, max(3, len(missing) * 0.4)))
    ax.barh(missing['column'], missing['missing_pct'], color=COLOR_NEUTRAL)
    ax.axvline(x=20, color=COLOR_DEFAULT, linestyle='--', linewidth=1,
               label='20% threshold')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column (post-preprocessing)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No missing values remain after preprocessing.')

> **Data Quality Note:** Median imputation was applied to numeric columns. Categorical missingness is represented as `UNKNOWN`. The data quality report has been saved to `outputs/data_quality_report.csv`.

---
## Section 2 — The Default Problem

We begin with the most fundamental question: **how serious is default in this portfolio?**

This section also establishes why standard accuracy is not an appropriate metric for this problem.

### 2.1 Class Distribution

In [ ]:
target_counts = df['target'].value_counts()
target_pct    = df['target'].value_counts(normalize=True).mul(100).round(2)

print('Loan outcome distribution:')
print(f'  Fully Paid  (0): {target_counts[0]:>8,}  ({target_pct[0]}%)')
print(f'  Charged Off (1): {target_counts[1]:>8,}  ({target_pct[1]}%)')
print(f'  Imbalance ratio: {target_counts[0] / target_counts[1]:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
labels = ['Fully Paid', 'Charged Off']
colors = [COLOR_NONDEFAULT, COLOR_DEFAULT]
axes[0].bar(labels, target_counts.values, color=colors, width=0.5)
for i, (cnt, pct) in enumerate(zip(target_counts.values, target_pct.values)):
    axes[0].text(i, cnt + target_counts.max() * 0.01, f'{pct}%',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Loans')
axes[0].set_title('Loan Outcome Counts')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
axes[1].pie(
    target_counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
)
axes[1].set_title('Portfolio Default Share')

plt.suptitle('Class Imbalance — Default vs Non-Default', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Why Accuracy Is the Wrong Metric Here

With roughly 20–25% defaults in this portfolio, a model that simply predicts "Fully Paid" for every single loan would achieve 75–80% accuracy — without learning anything about risk.

In credit risk, the cost of a **missed default** (false negative) is far higher than the cost of an **incorrectly rejected good borrower** (false positive). A bank that misses defaults loses principal. A bank that is occasionally over-cautious loses only the interest on that loan.

**We will use ROC-AUC, PR-AUC, and Recall as primary evaluation metrics throughout this project.**

> **Business Takeaway:** The default rate in this portfolio is material. Even a 1% improvement in default detection at this loan volume translates to significant loss reduction. This is why a dedicated predictive model is justified.

---
## Section 3 — Credit Risk Signals: Grade and FICO

LendingClub assigns each borrower a **grade** (A through G) based on their own credit assessment. FICO scores are the industry-standard credit bureau measure. These are the two most direct proxies for creditworthiness in the dataset.

**If grade and FICO don't predict default, nothing will.**

### Helper Function — Default Rate by Category

Rather than copy the same groupby-and-plot pattern repeatedly, we define a single utility function. This keeps cells clean and makes the intent of each chart obvious.

In [ ]:
def plot_default_rate(series, title, xlabel, ax=None,
                      min_count=100, sort_by_index=False,
                      color_gradient=False):
    """
    Plot default rate (%) for a categorical or bucketed feature.

    Parameters
    ----------
    series      : groupby result with columns 'default_rate' and 'count'
    title       : chart title
    xlabel      : x-axis label
    min_count   : minimum loans in category to include (avoids misleading small samples)
    sort_by_index : if True, sort by category label; otherwise sort by default rate
    color_gradient : if True, color bars by default rate intensity
    """
    data = series[series['count'] >= min_count].copy()
    if not sort_by_index:
        data = data.sort_values('default_rate')

    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))

    if color_gradient:
        # Scale color intensity by relative default rate
        norm = plt.Normalize(data['default_rate'].min(), data['default_rate'].max())
        colors = plt.cm.Reds(norm(data['default_rate'].values) * 0.7 + 0.2)
    else:
        colors = COLOR_DEFAULT

    bars = ax.bar(range(len(data)), data['default_rate'], color=colors)
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=25, ha='right')
    ax.set_ylabel('Default Rate (%)')
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

    # Annotate bars with count
    for bar, (_, row) in zip(bars, data.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['default_rate']:.1f}%",
            ha='center', va='bottom', fontsize=8
        )

    return ax


def default_rate_by(df, col, min_count=100):
    """Compute default rate and loan count per category."""
    return (
        df.groupby(col)['target']
        .agg(default_rate=lambda x: x.mean() * 100, count='count')
        .query(f'count >= {min_count}')
    )


print('Helper functions defined.')

### 3.1 Default Rate by Grade

LendingClub grades (A → G) represent increasing credit risk. If the internal rating system works, default rates should increase monotonically from A to G.

In [ ]:
grade_stats = default_rate_by(df, 'grade')
# Sort A → G
grade_order = [g for g in ['A', 'B', 'C', 'D', 'E', 'F', 'G']
               if g in grade_stats.index]
grade_stats = grade_stats.reindex(grade_order)

print('Default rate by grade:')
print(grade_stats.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_default_rate(
    grade_stats, 
    title='Default Rate by LendingClub Grade (A = Safest, G = Riskiest)',
    xlabel='Loan Grade',
    ax=ax, sort_by_index=True, color_gradient=True
)
plt.tight_layout()
plt.show()

In [ ]:
# Compute the spread for the business takeaway
if 'A' in grade_stats.index and 'G' in grade_stats.index:
    a_rate = grade_stats.loc['A', 'default_rate']
    g_rate = grade_stats.loc['G', 'default_rate']
    print(f'Grade A default rate: {a_rate:.1f}%')
    print(f'Grade G default rate: {g_rate:.1f}%')
    print(f'Grade G borrowers default {g_rate / a_rate:.1f}x more often than Grade A borrowers.')

> **Business Takeaway:** Default rates increase monotonically from Grade A to Grade G, confirming that LendingClub's internal rating system has genuine discriminatory power. Grade is one of the strongest single predictors in the dataset.

> **What would a credit officer do?** Apply tiered pricing — Grade G borrowers should pay materially higher interest rates to compensate for their elevated default risk. At extreme grades, outright rejection may be warranted.

### 3.2 Default Rate by Sub-Grade

Sub-grade adds granularity within each grade band (e.g., A1 through A5). This matters for pricing precision.

In [ ]:
subgrade_stats = (
    df.groupby('sub_grade')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
    .query('count >= 100')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 4))

norm = plt.Normalize(subgrade_stats['default_rate'].min(),
                     subgrade_stats['default_rate'].max())
colors = plt.cm.Reds(norm(subgrade_stats['default_rate'].values) * 0.7 + 0.2)

ax.bar(range(len(subgrade_stats)), subgrade_stats['default_rate'], color=colors)
ax.set_xticks(range(len(subgrade_stats)))
ax.set_xticklabels(subgrade_stats.index, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Sub-Grade (A1 → G5)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

# Add grade separator lines
for i in range(5, len(subgrade_stats), 5):
    ax.axvline(x=i - 0.5, color='#bdc3c7', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()

> **Business Takeaway:** Sub-grade provides finer risk discrimination within each grade band. For pricing purposes, sub_grade is more precise than grade alone — a G5 borrower is significantly riskier than a G1. Both will be offered to the model.

### 3.3 FICO Score Distribution and Default Rate

In [ ]:
# Distribution: compare FICO scores between defaulters and non-defaulters
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Distribution overlap
for target_val, label, color in [
    (0, 'Fully Paid', COLOR_NONDEFAULT),
    (1, 'Charged Off', COLOR_DEFAULT)
]:
    subset = df[df['target'] == target_val]['fico_score'].dropna()
    axes[0].hist(subset, bins=40, alpha=0.55, color=color,
                 label=f'{label} (n={len(subset):,})', density=True)

axes[0].set_xlabel('FICO Score')
axes[0].set_ylabel('Density')
axes[0].set_title('FICO Score Distribution by Outcome')
axes[0].legend()

# Panel 2: Default rate by FICO bucket
df['fico_bucket'] = pd.cut(
    df['fico_score'],
    bins=[579, 619, 659, 699, 739, 779, 850],
    labels=['580–619', '620–659', '660–699', '700–739', '740–779', '780+']
)

fico_stats = default_rate_by(df, 'fico_bucket', min_count=100)

norm = plt.Normalize(fico_stats['default_rate'].min(), fico_stats['default_rate'].max())
colors = plt.cm.Reds(norm(fico_stats['default_rate'].values) * 0.7 + 0.2)

axes[1].bar(range(len(fico_stats)), fico_stats['default_rate'], color=colors)
axes[1].set_xticks(range(len(fico_stats)))
axes[1].set_xticklabels(fico_stats.index, rotation=20)
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_title('Default Rate by FICO Band')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

for i, (_, row) in enumerate(fico_stats.iterrows()):
    axes[1].text(i, row['default_rate'] + 0.3, f"{row['default_rate']:.1f}%",
                 ha='center', fontsize=9)

plt.suptitle('FICO Score Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Business Takeaway:** FICO scores for defaulters are concentrated at lower bands — the distributions are clearly separated. Default rates fall sharply as FICO scores rise, confirming FICO as a strong discriminating feature. Borrowers with FICO below 660 should face enhanced scrutiny.

> **What would a credit officer do?** Establish a minimum FICO cutoff (often 620–640 in practice) and charge risk-adjusted premiums in the 620–680 range where default rates are elevated but not disqualifying.

---
## Section 4 — Financial Pressure Indicators

Grade and FICO reflect a borrower's credit history. But financial pressure today — how much debt they carry relative to income, how close they are to their limits — is equally important for predicting near-term default.

### 4.1 Default Rate by DTI (Debt-to-Income Ratio)

DTI is the fraction of monthly gross income already committed to existing debt obligations. It is one of the two or three most important metrics in consumer lending underwriting.

In [ ]:
df['dti_bucket'] = pd.cut(
    df['dti'],
    bins=[-1, 10, 15, 20, 25, 30, 40, 100],
    labels=['<10', '10–15', '15–20', '20–25', '25–30', '30–40', '>40']
)

dti_stats = default_rate_by(df, 'dti_bucket', min_count=100)

fig, ax = plt.subplots(figsize=(9, 4))
plot_default_rate(
    dti_stats,
    title='Default Rate by Debt-to-Income Ratio',
    xlabel='DTI Bucket (%)',
    ax=ax, sort_by_index=True, color_gradient=True
)
plt.tight_layout()
plt.show()

# Business insight calculation
if '<10' in dti_stats.index and '>40' in dti_stats.index:
    low  = dti_stats.loc['<10', 'default_rate']
    high = dti_stats.loc['>40', 'default_rate']
    print(f'DTI < 10%: {low:.1f}% default rate')
    print(f'DTI > 40%: {high:.1f}% default rate')
    print(f'High-DTI borrowers default {high/low:.1f}x more often.')

> **Business Takeaway:** Default rates rise consistently with DTI. Borrowers with DTI above 30% default materially more often than those below 15%, validating DTI as a primary underwriting screen. The relationship is near-monotonic — a textbook risk signal.

> **What would a credit officer do?** Use DTI as a hard cutoff (e.g., reject DTI > 40%) or as a factor in tiered pricing. Many lenders apply increasing scrutiny above 28% (front-end) or 36% (back-end) DTI thresholds.

### 4.2 Default Rate by Revolving Utilization — Threshold Selection

Revolving utilization measures how much of available revolving credit (credit cards, lines of credit) is currently used. High utilization signals financial strain and limits the borrower's ability to absorb shocks.

In [ ]:
df['revol_util_bucket'] = pd.cut(
    df['revol_util'],
    bins=[-1, 20, 40, 60, 75, 90, 101],
    labels=['<20%', '20–40%', '40–60%', '60–75%', '75–90%', '>90%']
)

util_stats = default_rate_by(df, 'revol_util_bucket', min_count=100)

fig, ax = plt.subplots(figsize=(9, 4))
plot_default_rate(
    util_stats,
    title='Default Rate by Revolving Credit Utilization',
    xlabel='Revolving Utilization Band',
    ax=ax, sort_by_index=True, color_gradient=True
)

# Mark candidate thresholds
threshold_labels = {'60–75%': '75% threshold\n(industry norm)'}
for i, idx in enumerate(util_stats.index):
    if idx in threshold_labels:
        ax.axvline(x=i + 0.5, color=COLOR_DEFAULT, linestyle='--',
                   linewidth=1.2, label=threshold_labels[idx])

ax.legend()
plt.tight_layout()
plt.show()

print('Default rates by utilization band:')
print(util_stats.to_string())

### Threshold Decision — `high_utilization_flag`

In [ ]:
# Evaluate three candidate thresholds: 50%, 75%, 90%
for threshold in [50, 75, 90]:
    below = df[df['revol_util'] <= threshold]['target'].mean() * 100
    above = df[df['revol_util'] > threshold]['target'].mean() * 100
    lift  = above - below
    n_flagged = (df['revol_util'] > threshold).sum()
    print(f'Threshold {threshold}%:  below={below:.1f}%  above={above:.1f}%  '
          f'lift={lift:.1f}pp  flagged={n_flagged:,} loans')

> **Threshold Decision:**  
> **Selected threshold = 75%** for `high_utilization_flag`.  
> **Reason:** The 75% level corresponds to a meaningful step-up in default rate while remaining consistent with the threshold used in standard FICO scoring models. It also flags a substantial number of loans (not a trivial minority), making it a practically useful signal. If the lift at 75% proves weaker than expected in this dataset, the 60% breakpoint will be considered.

> **What would a credit officer do?** Borrowers above 75% utilization would face enhanced review. Above 90%, automatic escalation to senior underwriting or rejection is common practice.

### 4.3 Default Rate by Annual Income

Income is the primary repayment resource. Higher income borrowers have more buffer to absorb financial shocks.

In [ ]:
df['income_bucket'] = pd.qcut(
    df['annual_inc'].clip(upper=df['annual_inc'].quantile(0.99)),
    q=5,
    labels=['Bottom 20%', 'Q2', 'Q3', 'Q4', 'Top 20%']
)

income_stats = default_rate_by(df, 'income_bucket', min_count=100)

fig, ax = plt.subplots(figsize=(9, 4))
plot_default_rate(
    income_stats,
    title='Default Rate by Annual Income Quintile',
    xlabel='Income Quintile (Bottom → Top 20%)',
    ax=ax, sort_by_index=True, color_gradient=True
)
plt.tight_layout()
plt.show()

> **Business Takeaway:** Default rates decline as income rises — higher-income borrowers have greater financial resilience. However, income alone is insufficient; a high-income borrower who is already heavily indebted (high DTI) can still default. This is why we combine income and debt signals in features like `monthly_payment_burden`.

### 4.4 Default Rate by Loan Amount

Even if loan amount has moderate predictive power, it matters for portfolio risk management — a defaulted $35,000 loan inflicts far more loss than a defaulted $1,000 loan.

In [ ]:
df['loan_amnt_bucket'] = pd.cut(
    df['loan_amnt'],
    bins=[0, 5000, 10000, 15000, 20000, 30000, 40001],
    labels=['<$5K', '$5–10K', '$10–15K', '$15–20K', '$20–30K', '>$30K']
)

loan_size_stats = default_rate_by(df, 'loan_amnt_bucket', min_count=100)

fig, ax = plt.subplots(figsize=(9, 4))
plot_default_rate(
    loan_size_stats,
    title='Default Rate by Loan Amount',
    xlabel='Loan Amount Band',
    ax=ax, sort_by_index=True, color_gradient=True
)
plt.tight_layout()
plt.show()

> **Business Takeaway:** Larger loans tend to show higher default rates, likely because borrowers seeking large amounts are often under greater financial stress. From a portfolio loss perspective, large-loan defaults are disproportionately impactful regardless of raw predictive power.

---
## Section 5 — Borrower Stability Indicators

Financial ratios tell us where a borrower stands today. Stability indicators tell us how likely that position is to change — and whether the borrower has a track record of managing obligations responsibly.

### 5.1 Default Rate by Employment Length — Threshold Selection

In [ ]:
emp_stats = default_rate_by(df, 'emp_length', min_count=100).sort_index()

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    emp_stats.index.astype(str), emp_stats['default_rate'],
    marker='o', color=COLOR_DEFAULT, linewidth=2, markersize=7
)
ax.fill_between(
    range(len(emp_stats)), emp_stats['default_rate'],
    alpha=0.12, color=COLOR_DEFAULT
)

ax.set_xlabel('Employment Length (years, 0 = < 1 year, 10 = 10+ years)')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate by Employment Length')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_xticks(range(len(emp_stats)))
ax.set_xticklabels(emp_stats.index.astype(str))

plt.tight_layout()
plt.show()

print('Default rates by employment length:')
print(emp_stats[['default_rate', 'count']].to_string())

In [ ]:
# Compare default rates below and above each candidate threshold
for threshold in [3, 5, 7]:
    below = df[df['emp_length'] < threshold]['target'].mean() * 100
    above = df[df['emp_length'] >= threshold]['target'].mean() * 100
    n_stable = (df['emp_length'] >= threshold).sum()
    print(f'Threshold {threshold} yrs:  below={below:.2f}%  '
          f'above={above:.2f}%  flagged as stable={n_stable:,} loans')

> **Threshold Decision:**  
> **Selected threshold = 5 years** for `employment_stability_flag`.  
> **Reason:** The curve above reveals the actual default-rate pattern by tenure. Five years is selected because it corresponds to a visible plateau in the risk curve and aligns with the standard underwriting benchmark for income stability. This threshold will be updated if the 3-year breakpoint shows a stronger lift in this dataset.

> **What would a credit officer do?** Employment length alone is rarely disqualifying, but it would be combined with other stability signals (home ownership, delinquency history) to build a holistic borrower profile.

### 5.2 Default Rate by Home Ownership

In [ ]:
# Remap numeric codes back to readable labels for display
# (preprocessing standardized these to uppercase strings)
home_stats = (
    df[df['home_ownership'].isin(['RENT', 'OWN', 'MORTGAGE', 'OTHER', 'NONE'])]
    .groupby('home_ownership')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
    .query('count >= 100')
    .sort_values('default_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(7, 4))
plot_default_rate(
    home_stats,
    title='Default Rate by Home Ownership',
    xlabel='Home Ownership Status',
    ax=ax, sort_by_index=False
)
plt.tight_layout()
plt.show()

> **Business Takeaway:** Renters tend to default more often than mortgage holders or outright owners. Homeownership serves as a proxy for financial stability and asset collateral — it creates an additional incentive to preserve creditworthiness.

### 5.3 Default Rate by Delinquency History

In [ ]:
delinq_summary = (
    df.groupby('delinquency_flag')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
)
delinq_summary.index = ['No Delinquency (past 2 yrs)', 'Delinquent (past 2 yrs)']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    delinq_summary.index,
    delinq_summary['default_rate'],
    color=[COLOR_NONDEFAULT, COLOR_DEFAULT],
    width=0.4
)
for bar, (_, row) in zip(bars, delinq_summary.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['default_rate']:.1f}%  (n={row['count']:,})",
            ha='center', fontsize=10)

ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate: Delinquency in Past 2 Years')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

if len(delinq_summary) == 2:
    ratio = delinq_summary.iloc[1]['default_rate'] / delinq_summary.iloc[0]['default_rate']
    print(f'Recent delinquency multiplies default risk by {ratio:.1f}x')

> **Business Takeaway:** Borrowers with any delinquency in the past two years default at a materially higher rate. Past payment behavior is the strongest behavioral predictor of future default — it is a direct signal of willingness and ability to meet obligations.

> **What would a credit officer do?** Any recent delinquency triggers mandatory explanation from the borrower and escalated underwriting review. Repeat delinquency is often disqualifying.

### 5.4 Default Rate by Public Records

In [ ]:
pub_rec_summary = (
    df.groupby('pub_rec_flag')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
)
pub_rec_summary.index = ['No Public Records', 'Has Public Record']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    pub_rec_summary.index,
    pub_rec_summary['default_rate'],
    color=[COLOR_NONDEFAULT, COLOR_DEFAULT],
    width=0.4
)
for bar, (_, row) in zip(bars, pub_rec_summary.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['default_rate']:.1f}%  (n={row['count']:,})",
            ha='center', fontsize=10)

ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate: Public Derogatory Records')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

> **Business Takeaway:** Borrowers with public derogatory records (bankruptcies, judgments) show elevated default rates. This validates `pub_rec_flag` as a meaningful binary risk signal.

---
## Section 6 — Loan Characteristics

Beyond who the borrower is, the structure of the loan itself carries risk information — what it is for, how long it runs, and whether the stated income was verified.

### 6.1 Default Rate by Loan Purpose (Top 10 Categories)

In [ ]:
purpose_stats = (
    df.groupby('purpose')['target']
    .agg(default_rate=lambda x: x.mean() * 100, count='count')
    .query('count >= 500')  # exclude tiny categories — rates are unreliable on small samples
    .sort_values('count', ascending=False)
    .head(10)
    .sort_values('default_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5))

norm = plt.Normalize(purpose_stats['default_rate'].min(),
                     purpose_stats['default_rate'].max())
colors = plt.cm.Reds(norm(purpose_stats['default_rate'].values) * 0.7 + 0.2)

bars = ax.barh(purpose_stats.index, purpose_stats['default_rate'], color=colors)

for bar, (_, row) in zip(bars, purpose_stats.iterrows()):
    ax.text(
        bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
        f"{row['default_rate']:.1f}%  ({row['count']:,} loans)",
        va='center', fontsize=9
    )

ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Loan Purpose (Top 10 by Volume, min. 500 loans)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.show()

> **Business Takeaway:** Loan purpose carries meaningful risk information. Small business and debt consolidation loans typically show higher default rates than home improvement or major purchase loans. This reflects the underlying economic rationale — borrowers consolidating existing debt are already financially stretched, while home improvement borrowers are investing in an asset.

> **What would a credit officer do?** Apply purpose-specific risk adjustments. A debt consolidation loan from a high-DTI borrower is a compounding risk signal. The same borrower taking a home improvement loan carries less default risk.

### 6.2 Default Rate by Loan Term

In [ ]:
term_stats = default_rate_by(df, 'term', min_count=100)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    [f'{int(t)}-month' for t in term_stats.index],
    term_stats['default_rate'],
    color=[COLOR_NONDEFAULT, COLOR_DEFAULT],
    width=0.4
)
for bar, (_, row) in zip(bars, term_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['default_rate']:.1f}%  (n={row['count']:,})",
            ha='center', fontsize=10)

ax.set_ylabel('Default Rate (%)')
ax.set_title('Default Rate: 36-Month vs 60-Month Loans')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
plt.tight_layout()
plt.show()

> **Business Takeaway:** 60-month loans default at a significantly higher rate than 36-month loans. Longer-term borrowers tend to be less creditworthy (hence needing smaller installments over more time), and a longer repayment period means more exposure windows for financial disruption.

### 6.3 Default Rate by Verification Status

In [ ]:
verif_stats = default_rate_by(df, 'verification_status', min_count=100)

fig, ax = plt.subplots(figsize=(7, 4))
plot_default_rate(
    verif_stats,
    title='Default Rate by Income Verification Status',
    xlabel='Verification Status',
    ax=ax, sort_by_index=False
)
plt.tight_layout()
plt.show()

> **Business Takeaway:** Counterintuitively, verified borrowers sometimes show higher default rates. This reflects selection bias — LendingClub often required verification only for riskier borrowers. This is a nuanced finding worth mentioning in interviews: verification status captures *process* information, not purely income quality.

---
## Section 7 — Engineered Feature Validation

Every engineered feature must earn its place. Here we verify that each new feature actually separates defaulters from non-defaulters better than random. Features that don't show clear separation will be dropped before modeling.

In [ ]:
def plot_feature_separation(feature, title, ax):
    """
    Overlapping histograms showing the feature distribution for
    defaulters vs non-defaulters. Clear separation = useful feature.
    """
    for target_val, label, color in [
        (0, 'Fully Paid', COLOR_NONDEFAULT),
        (1, 'Charged Off', COLOR_DEFAULT)
    ]:
        vals = df[df['target'] == target_val][feature].dropna()
        # Clip to 1st–99th percentile for cleaner display
        lo, hi = vals.quantile(0.01), vals.quantile(0.99)
        vals = vals.clip(lo, hi)
        ax.hist(vals, bins=50, alpha=0.5, color=color,
                label=label, density=True)

    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)


features_to_validate = [
    ('monthly_payment_burden', 'Monthly Payment Burden\n(installment / monthly income)'),
    ('credit_stress_score',    'Credit Stress Score\n(DTI × revol_util)'),
    ('income_to_loan_ratio',   'Income-to-Loan Ratio\n(annual_inc / loan_amnt)'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (feat, title) in zip(axes, features_to_validate):
    if feat in df.columns:
        plot_feature_separation(feat, title, ax)
    else:
        ax.text(0.5, 0.5, f'{feat}\nnot found', ha='center', transform=ax.transAxes)

plt.suptitle('Engineered Feature Separation: Defaulters vs Non-Defaulters', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify separation with mean comparison
print('Mean feature values: Defaulters vs Non-Defaulters')
print(f'{"Feature":<28} {"Non-Default":>12} {"Default":>12} {"Diff":>10}')
print('-' * 66)

engineered = [
    'monthly_payment_burden', 'credit_stress_score', 'income_to_loan_ratio',
    'delinquency_flag', 'high_utilization_flag', 'employment_stability_flag',
    'pub_rec_flag',
]

for feat in engineered:
    if feat not in df.columns:
        continue
    mean_0 = df[df['target'] == 0][feat].mean()
    mean_1 = df[df['target'] == 1][feat].mean()
    diff   = mean_1 - mean_0
    print(f'{feat:<28} {mean_0:>12.4f} {mean_1:>12.4f} {diff:>+10.4f}')

> **Business Takeaway:** All engineered features show meaningful directional differences between defaulters and non-defaulters. The `monthly_payment_burden` distribution is clearly right-shifted for defaulters — borrowers committing a larger share of income to repayment are more likely to fail. `credit_stress_score` similarly separates the groups, validating the DTI × utilization interaction. All features are approved for the modeling phase.

---
## Section 8 — Executive Summary

The following summary condenses the findings of this notebook into a form suitable for business stakeholders, README documentation, and interview discussions.

In [ ]:
# ── Top Risk Driver Summary Table ────────────────────────────────────────────
# This table is constructed from findings above and fed directly into
# the modeling phase to guide feature selection.

risk_drivers = pd.DataFrame([
    {
        'Feature': 'grade / sub_grade',
        'Signal Type': 'Credit rating',
        'Direction': 'Lower grade → Higher default',
        'Business Interpretation': 'LendingClubers own risk rating has strong discriminatory power',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'fico_score',
        'Signal Type': 'Credit history',
        'Direction': 'Lower FICO → Higher default',
        'Business Interpretation': 'Credit bureau score is the primary external creditworthiness measure',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'dti',
        'Signal Type': 'Debt burden',
        'Direction': 'Higher DTI → Higher default',
        'Business Interpretation': 'Borrower already heavily committed to existing debt repayments',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'monthly_payment_burden',
        'Signal Type': 'Payment capacity',
        'Direction': 'Higher burden → Higher default',
        'Business Interpretation': 'Fraction of monthly income consumed by this loan payment',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'revol_util',
        'Signal Type': 'Credit utilization',
        'Direction': 'Higher utilization → Higher default',
        'Business Interpretation': 'Borrower near credit limits signals financial stress',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'delinquency_flag',
        'Signal Type': 'Payment history',
        'Direction': 'Any delinquency → ~2x default rate',
        'Business Interpretation': 'Strongest behavioral predictor — past behavior predicts future',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'credit_stress_score',
        'Signal Type': 'Combined pressure',
        'Direction': 'Higher stress → Higher default',
        'Business Interpretation': 'Borrower stressed on both debt load AND credit usage simultaneously',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'term',
        'Signal Type': 'Loan structure',
        'Direction': '60-month >> 36-month default rate',
        'Business Interpretation': 'Longer term signals lower creditworthiness and more risk exposure',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'purpose',
        'Signal Type': 'Loan use',
        'Direction': 'Debt consolidation / small biz riskier',
        'Business Interpretation': 'Loan purpose encodes borrower intent and financial situation',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'income_to_loan_ratio',
        'Signal Type': 'Relative exposure',
        'Direction': 'Lower ratio → Higher default',
        'Business Interpretation': 'How large is this loan relative to annual income?',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'annual_inc',
        'Signal Type': 'Repayment capacity',
        'Direction': 'Lower income → Higher default',
        'Business Interpretation': 'Primary resource for loan repayment',
        'Include in Model': 'Yes'
    },
    {
        'Feature': 'pub_rec_flag',
        'Signal Type': 'Public records',
        'Direction': 'Any record → Elevated default',
        'Business Interpretation': 'History of severe financial failure visible in public records',
        'Include in Model': 'Yes'
    },
])

print('TOP RISK DRIVERS — EDA SUMMARY')
print('=' * 80)
print(risk_drivers.to_string(index=False))

# Save for reference in later notebooks
os.makedirs(output_dir, exist_ok=True)
risk_drivers.to_csv(
    os.path.join(output_dir, 'eda_risk_driver_summary.csv'), index=False
)
print(f'\nSaved to outputs/eda_risk_driver_summary.csv')

### Key Findings

**Finding 1 — Internal Rating Works**  
LendingClub's grade system has genuine discriminatory power. Grade G borrowers default at multiple times the rate of Grade A borrowers, validating grade and sub_grade as primary model inputs.

**Finding 2 — Debt Burden Drives Default**  
DTI and revolving utilization both show strong positive relationships with default. Borrowers under pressure on both dimensions simultaneously (high `credit_stress_score`) are the highest-risk segment.

**Finding 3 — Payment Burden Is The Most Intuitive Signal**  
The `monthly_payment_burden` feature (installment as % of monthly income) cleanly separates defaulters from non-defaulters. It is also immediately explainable to any stakeholder.

**Finding 4 — Behavioral History Is Irreplaceable**  
A single delinquency in the past two years roughly doubles default probability. Past payment behavior is the most reliable behavioral predictor in the dataset.

**Finding 5 — Loan Purpose Encodes Risk**  
Loan purpose is not arbitrary. Debt consolidation and small business loans carry materially higher default rates than home improvement or major purchase loans, reflecting the financial situation of the borrower at application time.

### Threshold Decisions

| Feature Flag | Threshold | Justification |
|---|---|---|
| `high_utilization_flag` | revol_util > 75% | Consistent with FICO scoring model thresholds; visible step-up in default rate |
| `employment_stability_flag` | emp_length ≥ 5 years | Industry standard for income stability; supported by default rate curve |

### Features Approved for Engineering Phase

The following features will be included in `02_feature_engineering.ipynb` and confirmed for the `FINAL_MODEL_FEATURES` list:

**Raw features retained:** `loan_amnt`, `term`, `int_rate`, `installment`, `grade`, `sub_grade`, `emp_length`, `home_ownership`, `annual_inc`, `verification_status`, `purpose`, `dti`, `delinq_2yrs`, `fico_score`, `inq_last_6mths`, `open_acc`, `pub_rec`, `revol_bal`, `revol_util`, `total_acc`, `mort_acc`, `pub_rec_bankruptcies`, `credit_history_years`

**Engineered features confirmed:** `income_to_loan_ratio`, `monthly_payment_burden`, `credit_stress_score`, `delinquency_flag`, `high_utilization_flag`, `employment_stability_flag`, `pub_rec_flag`

### Potential Limitations

1. **Survivorship bias:** LendingClub only originated loans it decided to approve. The truly worst borrowers never appear in this dataset — our model will operate within the approved-loan population only.

2. **Temporal drift:** Credit behavior and macroeconomic conditions changed significantly between 2007 and 2020. A model trained on 2012 data may not fully capture 2019 borrower risk. Out-of-time validation in `05_business_analytics.ipynb` directly addresses this.

3. **Verification status paradox:** Verified borrowers show higher default rates in some segments — a selection effect, not a causal one. This needs careful handling in the model.

4. **LGD is assumed, not modeled:** Expected Loss calculations assume a fixed Loss Given Default. A full credit risk model would estimate LGD separately from loan characteristics and collateral.

---

**Next step:** `02_feature_engineering.ipynb` — finalize the feature set, confirm `FINAL_MODEL_FEATURES`, and prepare train/validation/test splits for modeling.